In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (classification_report, confusion_matrix, 
                             f1_score, matthews_corrcoef,
                             average_precision_score, roc_auc_score)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/train_final.csv')
test  = pd.read_csv('../data/test_final.csv')

X_train = train.drop(columns=['isFraud'])
y_train = train['isFraud']
X_test  = test.drop(columns=['isFraud'])
y_test  = test['isFraud']

print(f"Train: {X_train.shape}")
print(f"Test:  {X_test.shape}")

Train: (15624, 192)
Test:  (2000, 192)


In [3]:
def evaluate_model(name, y_true, y_pred, y_prob):
    f1  = f1_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    auprc = average_precision_score(y_true, y_prob)
    auc   = roc_auc_score(y_true, y_prob)
    
    print(f"\n{'='*40}")
    print(f"Model: {name}")
    print(f"{'='*40}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"MCC:       {mcc:.4f}")
    print(f"AUPRC:     {auprc:.4f}")
    print(f"ROC-AUC:   {auc:.4f}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['Legit','Fraud'])}")
    
    return {'model': name, 'F1': f1, 'MCC': mcc, 'AUPRC': auprc, 'AUC': auc}

In [4]:
print("Training Logistic Regression...")
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

results_lr = evaluate_model("Logistic Regression", y_test, y_pred_lr, y_prob_lr)

Training Logistic Regression...

Model: Logistic Regression
F1 Score:  0.1074
MCC:       0.0851
AUPRC:     0.0830
ROC-AUC:   0.6414

              precision    recall  f1-score   support

       Legit       0.97      0.62      0.76      1923
       Fraud       0.06      0.60      0.11        77

    accuracy                           0.62      2000
   macro avg       0.52      0.61      0.43      2000
weighted avg       0.94      0.62      0.73      2000



In [5]:
print("Training Decision Tree...")
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

results_dt = evaluate_model("Decision Tree", y_test, y_pred_dt, y_prob_dt)

Training Decision Tree...

Model: Decision Tree
F1 Score:  0.4421
MCC:       0.4238
AUPRC:     0.2236
ROC-AUC:   0.7415

              precision    recall  f1-score   support

       Legit       0.98      0.96      0.97      1923
       Fraud       0.37      0.55      0.44        77

    accuracy                           0.95      2000
   macro avg       0.68      0.75      0.71      2000
weighted avg       0.96      0.95      0.95      2000



In [6]:
results = [results_lr, results_dt]
df_results = pd.DataFrame(results)

print("\n📊 Bảng so sánh baseline:")
print(df_results.to_string(index=False))

# Vẽ biểu đồ so sánh
metrics = ['F1', 'MCC', 'AUPRC', 'AUC']
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, df_results.iloc[0][metrics], width, label='Logistic Regression', color='steelblue')
ax.bar(x + width/2, df_results.iloc[1][metrics], width, label='Decision Tree', color='coral')

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('Score')
ax.set_title('Baseline Models Comparison')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../reports/baseline_comparison.png', dpi=80, bbox_inches='tight')
plt.close()

df_results.to_csv('../reports/baseline_results.csv', index=False)
print("\n✅ Đã lưu kết quả!")


📊 Bảng so sánh baseline:
              model       F1      MCC    AUPRC      AUC
Logistic Regression 0.107351 0.085089 0.083044 0.641409
      Decision Tree 0.442105 0.423769 0.223564 0.741509

✅ Đã lưu kết quả!
